# DeepLearning.ai - [Knowledge Graphs for RAG](https://www.deeplearning.ai/courses/knowledge-graphs-rag)

## Lesson 2: Querying Knowledge Graphs with Cypher

### Import packages and set up Neo4


In [10]:
from dotenv import load_dotenv
import hashlib
import os
import neo4j

from langchain_community.graphs import Neo4jGraph

# Warning control
import warnings
warnings.filterwarnings("ignore")

In [51]:
load_dotenv('.env', override=True)
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

print(NEO4J_URI)
print(NEO4J_DATABASE)
print(NEO4J_USERNAME)
print(hashlib.sha256(NEO4J_PASSWORD.encode()).hexdigest()[:8])
# printf '%s' "$NEO4J_PASSWORD" | sha256sum | cut -c1-8

bolt://172.30.2.177:7687
neo4j
neo4j
b90caf5c


In [52]:
kg = Neo4jGraph(
    url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE
)

### Querying the movie knowledge graph 
- Match all nodes in the graph

In [53]:
cypher = """
  MATCH (n) 
  RETURN count(n)
  """

In [54]:
result = kg.query(cypher)
result

[{'count(n)': 173}]

In [7]:
cypher = """
  MATCH (n) 
  RETURN count(n) AS numberOfNodes
  """

In [8]:
result = kg.query(cypher)
result

[{'numberOfNodes': 171}]

In [9]:
print(f"There are {result[0]['numberOfNodes']} nodes in this graph.")

There are 171 nodes in this graph.


- Match only the Movie nodes by specifying the node label

In [10]:
cypher = """
  MATCH (n:Movie) 
  RETURN count(n) AS numberOfMovies
  """
kg.query(cypher)

[{'numberOfMovies': 38}]

- Change the variable name in the node pattern match for improved readability

In [11]:
cypher = """
  MATCH (m:Movie) 
  RETURN count(m) AS numberOfMovies
  """
kg.query(cypher)

[{'numberOfMovies': 38}]

- Match only the Person nodes

In [12]:
cypher = """
  MATCH (people:Person) 
  RETURN count(people) AS numberOfPeople
  """
kg.query(cypher)

[{'numberOfPeople': 133}]

- Match a single person by specifying the value of the name property on the Person node

In [13]:
cypher = """
  MATCH (tom:Person {name:"Tom Hanks"}) 
  RETURN tom
  """
kg.query(cypher)

[{'tom': {'born': 1956, 'name': 'Tom Hanks'}}]

- Match a single Movie by specifying the value of the title property

In [14]:
cypher = """
  MATCH (cloudAtlas:Movie {title:"Cloud Atlas"}) 
  RETURN cloudAtlas
  """
kg.query(cypher)

[{'cloudAtlas': {'tagline': 'Everything is connected',
   'title': 'Cloud Atlas',
   'released': 2012}}]

- Return only the released property of the matched Movie node

In [15]:
cypher = """
  MATCH (cloudAtlas:Movie {title:"Cloud Atlas"}) 
  RETURN cloudAtlas.released
  """
kg.query(cypher)

[{'cloudAtlas.released': 2012}]

- Return two properties

In [16]:
cypher = """
  MATCH (cloudAtlas:Movie {title:"Cloud Atlas"}) 
  RETURN cloudAtlas.released, cloudAtlas.tagline
  """
kg.query(cypher)

[{'cloudAtlas.released': 2012,
  'cloudAtlas.tagline': 'Everything is connected'}]

### Cypher patterns with conditional matching

In [17]:
cypher = """
  MATCH (nineties:Movie) 
  WHERE nineties.released >= 1990 
    AND nineties.released < 2000 
  RETURN nineties.title
  """
kg.query(cypher)

[{'nineties.title': 'Joe Versus the Volcano'},
 {'nineties.title': 'A Few Good Men'},
 {'nineties.title': 'Unforgiven'},
 {'nineties.title': 'Hoffa'},
 {'nineties.title': 'A League of Their Own'},
 {'nineties.title': 'Sleepless in Seattle'},
 {'nineties.title': 'Johnny Mnemonic'},
 {'nineties.title': 'Apollo 13'},
 {'nineties.title': 'That Thing You Do'},
 {'nineties.title': 'The Birdcage'},
 {'nineties.title': 'Twister'},
 {'nineties.title': "The Devil's Advocate"},
 {'nineties.title': 'As Good as It Gets'},
 {'nineties.title': 'What Dreams May Come'},
 {'nineties.title': "You've Got Mail"},
 {'nineties.title': 'When Harry Met Sally'},
 {'nineties.title': 'The Matrix'},
 {'nineties.title': 'Snow Falling on Cedars'},
 {'nineties.title': 'The Green Mile'},
 {'nineties.title': 'Bicentennial Man'}]

### Pattern matching with multiple nodes

In [22]:
cypher = """
  MATCH (actor:Person)-[:ACTED_IN]->(movie:Movie) 
  RETURN actor.name, movie.title LIMIT 10
  """
kg.query(cypher)

[{'actor.name': 'Hugo Weaving', 'movie.title': 'The Matrix'},
 {'actor.name': 'Laurence Fishburne', 'movie.title': 'The Matrix'},
 {'actor.name': 'Carrie-Anne Moss', 'movie.title': 'The Matrix'},
 {'actor.name': 'Keanu Reeves', 'movie.title': 'The Matrix'},
 {'actor.name': 'Hugo Weaving', 'movie.title': 'The Matrix Reloaded'},
 {'actor.name': 'Laurence Fishburne', 'movie.title': 'The Matrix Reloaded'},
 {'actor.name': 'Carrie-Anne Moss', 'movie.title': 'The Matrix Reloaded'},
 {'actor.name': 'Keanu Reeves', 'movie.title': 'The Matrix Reloaded'},
 {'actor.name': 'Hugo Weaving', 'movie.title': 'The Matrix Revolutions'},
 {'actor.name': 'Laurence Fishburne', 'movie.title': 'The Matrix Revolutions'}]

In [23]:
cypher = """
  MATCH (tom:Person {name: "Tom Hanks"})-[:ACTED_IN]->(tomHanksMovies:Movie) 
  RETURN tom.name,tomHanksMovies.title LIMIT 10
  """
kg.query(cypher)

[{'tom.name': 'Tom Hanks', 'tomHanksMovies.title': 'A League of Their Own'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovies.title': 'The Polar Express'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovies.title': "Charlie Wilson's War"},
 {'tom.name': 'Tom Hanks', 'tomHanksMovies.title': 'Cast Away'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovies.title': 'Apollo 13'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovies.title': 'The Green Mile'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovies.title': 'The Da Vinci Code'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovies.title': 'Cloud Atlas'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovies.title': 'That Thing You Do'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovies.title': 'Joe Versus the Volcano'}]

In [24]:
cypher = """
  MATCH (tom:Person {name:"Tom Hanks"})-[:ACTED_IN]->(m)<-[:ACTED_IN]-(coActors) 
  RETURN coActors.name, m.title LIMIT 10
  """
kg.query(cypher)

[{'coActors.name': 'Geena Davis', 'm.title': 'A League of Their Own'},
 {'coActors.name': 'Bill Paxton', 'm.title': 'A League of Their Own'},
 {'coActors.name': 'Madonna', 'm.title': 'A League of Their Own'},
 {'coActors.name': 'Lori Petty', 'm.title': 'A League of Their Own'},
 {'coActors.name': "Rosie O'Donnell", 'm.title': 'A League of Their Own'},
 {'coActors.name': 'Philip Seymour Hoffman',
  'm.title': "Charlie Wilson's War"},
 {'coActors.name': 'Julia Roberts', 'm.title': "Charlie Wilson's War"},
 {'coActors.name': 'Helen Hunt', 'm.title': 'Cast Away'},
 {'coActors.name': 'Ed Harris', 'm.title': 'Apollo 13'},
 {'coActors.name': 'Gary Sinise', 'm.title': 'Apollo 13'}]

### Create a vector index 

First, we need to get setup to use [Titan embeddings via Amazon Bedrock endpoint](https://neo4j.com/docs/genai/plugin/current/embeddings/):

In [18]:
import boto3

frozen = boto3.Session().get_credentials().get_frozen_credentials()   # default chain: ~/.aws
print("key id prefix:", frozen.access_key[:4])          # AKIA = long-lived, ASIA = temporary
print("session token present:", frozen.token is not None)

key id prefix: AKIA
session token present: False


In [25]:
aws_cfg = {
    "accessKeyId": frozen.access_key,
    "secretAccessKey": frozen.secret_key,
    "region": "us-east-1",
    "model": "amazon.titan-embed-text-v2:0",
}

kg.query("""
RETURN size(genai.vector.encode($text, 'Bedrock', $config)) AS dimensions
""", params={"text": "Welcome to the Real World", "config": aws_cfg})

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. genai.vector.encode is deprecated. It is replaced by ai.text.embed.', position=<SummaryInputPosition line=2, column=13, offset=13>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 13, 'line': 2, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nRETURN size(genai.vector.encode($text, 'Bedrock', $config)) AS dimensions\n"


[{'dimensions': 1024}]

In [48]:
kg.query("""
  CREATE VECTOR INDEX movie_tagline_embeddings IF NOT EXISTS
  FOR (m:Movie) ON (m.taglineEmbedding)
  OPTIONS { indexConfig: {
    `vector.dimensions`: 1024,
    `vector.similarity_function`: 'cosine'
  }}"""
)

# kg.query("DROP INDEX movie_tagline_embeddings IF EXISTS")

[]

In [49]:
kg.query("""
  SHOW VECTOR INDEXES
  """
)

[{'id': 9,
  'name': 'movie_tagline_embeddings',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'VECTOR',
  'entityType': 'NODE',
  'labelsOrTypes': ['Movie'],
  'properties': ['taglineEmbedding'],
  'indexProvider': 'vector-2026.08',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': None}]

In [28]:
kg.query("CALL dbms.components() YIELD name, versions, edition RETURN name, versions, edition")

[{'name': 'Neo4j Kernel', 'versions': ['2026.09.0'], 'edition': 'community'},
 {'name': 'Cypher', 'versions': ['5', '25'], 'edition': ''}]

In [29]:
kg.query("""
  SHOW FUNCTIONS YIELD name
  WHERE name STARTS WITH 'genai' OR name STARTS WITH 'ai.'
  RETURN name
""")

[{'name': 'genai.vector.encode'}]

In [30]:
kg.query("""
  CALL genai.vector.listEncodingProviders()
""")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. genai.vector.listEncodingProviders is deprecated. It is replaced by ai.text.embed.providers.', position=<SummaryInputPosition line=2, column=3, offset=3>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 3, 'line': 2, 'column': 3}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n  CALL genai.vector.listEncodingProviders()\n'


[{'name': 'AzureOpenAI',
  'requiredConfigType': '{ token :: STRING NOT NULL, resource :: STRING NOT NULL, deployment :: STRING NOT NULL }',
  'optionalConfigType': '{ dimensions :: INTEGER }',
  'defaultConfig': {}},
 {'name': 'Bedrock',
  'requiredConfigType': '{ accessKeyId :: STRING NOT NULL, secretAccessKey :: STRING NOT NULL }',
  'optionalConfigType': '{ model :: STRING NOT NULL, region :: STRING NOT NULL, dimensions :: INTEGER, normalize :: BOOLEAN }',
  'defaultConfig': {'region': 'us-east-1',
   'model': 'amazon.titan-embed-text-v1'}},
 {'name': 'OpenAI',
  'requiredConfigType': '{ token :: STRING NOT NULL }',
  'optionalConfigType': '{ model :: STRING NOT NULL, dimensions :: INTEGER }',
  'defaultConfig': {'model': 'text-embedding-ada-002'}},
 {'name': 'VertexAI',
  'requiredConfigType': '{ token :: STRING NOT NULL, projectId :: STRING NOT NULL }',
  'optionalConfigType': '{ model :: STRING NOT NULL, region :: STRING NOT NULL, taskType :: STRING, title :: STRING, autoTruncat

In [31]:
kg.query("""
  SHOW PROCEDURES YIELD name
  WHERE name STARTS WITH 'genai' OR name = 'db.create.setNodeVectorProperty'
  RETURN name
""")

[{'name': 'db.create.setNodeVectorProperty'},
 {'name': 'genai.vector.encodeBatch'},
 {'name': 'genai.vector.listEncodingProviders'}]

### Populate the vector index
- Calculate vector representation for each movie tagline using OpenAI
- Add vector to the `Movie` node as `taglineEmbedding` property


In [55]:
kg.query("""
    MATCH (movie:Movie) WHERE movie.tagline IS NOT NULL
    WITH movie, ai.text.embed(
        movie.tagline,
        'bedrock-titan',
        $config) AS vector
    CALL db.create.setNodeVectorProperty(movie, 'taglineEmbedding', vector)
    """,
    params={"config": aws_cfg}
    )

[]

In [56]:
result = kg.query("""
    MATCH (m:Movie) 
    WHERE m.tagline IS NOT NULL
    RETURN m.tagline, m.taglineEmbedding
    LIMIT 1
    """
)

In [57]:
result[0]['m.tagline']

'Welcome to the Real World'

In [58]:
len(result[0]['m.taglineEmbedding'])

1024

### Similarity search
- Calculate embedding for question
- Identify matching movies based on similarity of question and `taglineEmbedding` vectors

In [59]:
question = "What movies are about love?"

kg.query("""
    WITH ai.text.embed($question, 'bedrock-titan', $config) AS question_embedding
    MATCH (movie:Movie)
    SEARCH movie IN (
        VECTOR INDEX movie_tagline_embeddings
        FOR question_embedding
        LIMIT $top_k
    ) SCORE AS score
    RETURN movie.title, movie.tagline, score
    ORDER BY score DESC
    """,
    params={"config": aws_cfg,
            "question": question,
            "top_k": 5
            }
    )

[{'movie.title': 'Joe Versus the Volcano',
  'movie.tagline': 'A story of love, lava and burning desire.',
  'score': 0.6089553236961365},
 {'movie.title': 'Snow Falling on Cedars',
  'movie.tagline': 'First loves last. Forever.',
  'score': 0.5974831581115723},
 {'movie.title': "You've Got Mail",
  'movie.tagline': 'At odds in life... in love on-line.',
  'score': 0.5863548517227173},
 {'movie.title': 'Stand By Me',
  'movie.tagline': "For some, it's the last real taste of innocence, and the first real taste of life. But for everyone, it's the time that memories are made of.",
  'score': 0.5583323836326599},
 {'movie.title': 'As Good as It Gets',
  'movie.tagline': 'A comedy from the heart that goes for the throat.',
  'score': 0.5537737011909485}]

In [62]:
question = "What movies are about freedom?"

kg.query("""
    WITH ai.text.embed($question, 'bedrock-titan', $config) AS question_embedding
    MATCH (movie:Movie)
    SEARCH movie IN (
        VECTOR INDEX movie_tagline_embeddings
        FOR question_embedding
        LIMIT $top_k
    ) SCORE AS score
    RETURN movie.title, movie.tagline, score
    ORDER BY score DESC
    """,
    params={"config": aws_cfg,
            "question": question,
            "top_k": 5
            }
    )

[{'movie.title': 'RescueDawn',
  'movie.tagline': "Based on the extraordinary true story of one man's fight for freedom",
  'score': 0.6479344964027405},
 {'movie.title': 'V for Vendetta',
  'movie.tagline': 'Freedom! Forever!',
  'score': 0.6410220861434937},
 {'movie.title': 'The Matrix Reloaded',
  'movie.tagline': 'Free your mind',
  'score': 0.6005020141601562},
 {'movie.title': 'The Da Vinci Code',
  'movie.tagline': 'Break The Codes',
  'score': 0.5902088284492493},
 {'movie.title': 'Speed Racer',
  'movie.tagline': 'Speed has no limits',
  'score': 0.579997181892395}]